In [1]:
print("Hello")

Hello


# Preprocessing

In [1]:
import os
import re
import glob
import subprocess
import pandas as pd
from tqdm.notebook import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
ROOT_DIR = "./"

# UPDATED REGEX EXPLANATION:
# ^([0-9.]+)M\s+(\S+)       -> Matches "0.5M H2SO4" (Start)
# (?: ... )?                -> Non-capturing group for the optional suffix
# \s+Further\s+[0-9.]+      -> Matches " Further 1"
# |                         -> OR
# \s+[0-9.-]+V              -> Matches " 0.2V" or " -0.5V"
FOLDER_PATTERN = re.compile(r"^([0-9.]+)M\s+(\S+)(?:\s+Further\s+[0-9.]+|\s+[0-9.-]+V)?$")

def preprocess_data():
    # 1. Identify all valid folders first
    all_items = os.listdir(ROOT_DIR)
    valid_folders = []
    
    for item in all_items:
        # Check if it's a directory AND matches our naming convention
        if os.path.isdir(os.path.join(ROOT_DIR, item)) and FOLDER_PATTERN.match(item):
            valid_folders.append(item)
    
    print(f"📂 Found {len(valid_folders)} valid trial folders. Starting preprocessing...\n")

    # 2. Iterate through folders with a Progress Bar
    if not valid_folders:
        print("No folders found matching the naming convention.")
        return

    pbar = tqdm(valid_folders, unit="trial")
    
    for folder_name in pbar:
        pbar.set_description(f"Processing {folder_name}")
        folder_path = os.path.join(ROOT_DIR, folder_name)
        
        # logs for this specific folder
        status_cv = ""
        status_vid = ""

        # ---------------------------------------------------------
        # A. PROCESS CV DATA (.tsv -> .csv transposed)
        # ---------------------------------------------------------
        csv_out_path = os.path.join(folder_path, "cv_transposed.csv")
        
        if os.path.exists(csv_out_path):
            status_cv = "⏩ CV (Skipped)"
        else:
            # Find the .tsv file (there should be exactly 1)
            tsv_files = glob.glob(os.path.join(folder_path, "*.tsv"))
            
            if not tsv_files:
                status_cv = "❌ CV (No .tsv found)"
            else:
                try:
                    # Pick the first tsv found
                    tsv_path = tsv_files[0]
                    
                    # Read and Transpose
                    # sep='\t' is crucial for tsv
                    data_og = pd.read_csv(tsv_path, sep="\t")
                    
                    # Save transposed
                    data_og.T.to_csv(csv_out_path, index=True)
                    status_cv = "✅ CV (Transposed)"
                except Exception as e:
                    status_cv = f"⚠️ CV Error: {str(e)}"

        # ---------------------------------------------------------
        # B. PROCESS VIDEO (.avi -> .mp4 h.264)
        # ---------------------------------------------------------
        mp4_out_path = os.path.join(folder_path, "out.mp4")
        
        if os.path.exists(mp4_out_path):
            status_vid = "⏩ Video (Skipped)"
        else:
            # Find the raw video file
            raw_vid_files = glob.glob(os.path.join(folder_path, "*.avi"))
            
            if not raw_vid_files:
                status_vid = "❌ Video (No .avi found)"
            else:
                try:
                    # Pick the first video found
                    input_vid = raw_vid_files[0]
                    
                    # Construct FFmpeg command
                    cmd = [
                        'ffmpeg', 
                        '-y',                  # Overwrite output
                        '-loglevel', 'error',  # Suppress generic output
                        '-i', input_vid, 
                        '-c:v', 'libx264', 
                        '-preset', 'medium', 
                        '-crf', '23', 
                        mp4_out_path
                    ]
                    
                    # Run FFmpeg
                    subprocess.run(cmd, check=True)
                    status_vid = "✅ Video (Converted)"
                except subprocess.CalledProcessError:
                    status_vid = "⚠️ Video (FFmpeg Failed)"
                except FileNotFoundError:
                    status_vid = "⚠️ Video (FFmpeg not installed/found)"

        # Print summary for this folder
        tqdm.write(f"[{folder_name}]: {status_cv} | {status_vid}")

    print("\n✨ Preprocessing Complete!")

# Run the function
preprocess_data()

📂 Found 39 valid trial folders. Starting preprocessing...



  0%|          | 0/39 [00:00<?, ?trial/s]

[0.5M HCl Further 1]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M HCl Further 2]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M HCl Further 4]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M HCl Further 3]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M HCl]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M H2SO4]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M H2SO4 Further 1]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M H2SO4 Further 2]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M H2SO4 Further 3]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[0.5M H2SO4 Further 4]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M H2SO4]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M H2SO4 Further 1]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M H2SO4 Further 2]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M H2SO4 Further 3]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M H2SO4 Further 4]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M H2SO4 Further 5]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M HCl]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M HCl Further 1]: ⏩ CV (Skipped) | ⏩ Video (Skipped)
[2M HCl 

In [2]:
# Pre processing (Skip)
# data_og = pd.read_csv("./2M H2SO4/cv.tsv", sep="\t")
# data_og.T.to_csv("./2M H2SO4/cv_transposed.csv", index=True)

# Video processing
# ffmpeg -i .\test-12052025105132-0000.avi -c:v libx264 -preset medium -crf 23 out.mp4

# Video CV Comparison

In [9]:
import sys
import os
import pandas as pd
import numpy as np
import cv2
from PyQt6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                             QHBoxLayout, QLabel, QSlider, QPushButton, 
                             QLineEdit, QGroupBox, QFileDialog, QComboBox, QMessageBox)
from PyQt6.QtCore import Qt, QTimer
from PyQt6.QtGui import QImage, QPixmap
import pyqtgraph as pg

class ExperimentVisualizer(QMainWindow):
    def __init__(self):
        super().__init__()
        
        # State Variables
        self.df = None
        self.video_path = None
        self.cap = None
        self.time_offset = 0.0 
        
        # Data Arrays
        self.raw_times = np.array([])
        self.raw_voltage = np.array([])
        self.raw_current = np.array([])
        self.split_idx = 0
        
        # Playback State
        self.fps = 30.0
        self.total_frames = 0
        self.video_duration = 0
        self.ms_per_frame = 33.33
        self.is_playing = False
        
        # UI Setup
        self.init_ui()
        
        # Timer
        self.timer = QTimer()
        self.timer.timeout.connect(self.next_frame)

    def init_ui(self):
        self.setWindowTitle("Experiment Visualizer (CV & Constant V)")
        self.resize(1400, 900)
        
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QHBoxLayout(central_widget)
        
        # ==================== LEFT SIDE (Controls & Video) ====================
        left_panel = QVBoxLayout()
        
        # --- 1. FILE LOADING AREA ---
        file_group = QGroupBox("1. File Selection")
        file_layout = QFormLayout_Custom() # Helper layout below
        
        self.btn_load_csv = QPushButton("Load CSV...")
        self.btn_load_csv.clicked.connect(self.load_csv_dialog)
        self.lbl_csv_name = QLabel("No CSV loaded")
        self.lbl_csv_name.setStyleSheet("color: #666;")
        
        self.btn_load_vid = QPushButton("Load Video...")
        self.btn_load_vid.clicked.connect(self.load_video_dialog)
        self.lbl_vid_name = QLabel("No Video loaded")
        self.lbl_vid_name.setStyleSheet("color: #666;")
        
        file_layout.addRow(self.btn_load_csv, self.lbl_csv_name)
        file_layout.addRow(self.btn_load_vid, self.lbl_vid_name)
        file_group.setLayout(file_layout)
        left_panel.addWidget(file_group)

        # --- 2. VIDEO PLAYER ---
        self.video_label = QLabel("Please Load Video")
        self.video_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.video_label.setMinimumSize(640, 480)
        self.video_label.setStyleSheet("background-color: black; border: 1px solid #444; color: white;")
        left_panel.addWidget(self.video_label)
        
        self.slider = QSlider(Qt.Orientation.Horizontal)
        self.slider.setRange(0, 100) 
        self.slider.sliderMoved.connect(self.slider_moved)
        self.slider.sliderPressed.connect(self.slider_pressed)
        self.slider.sliderReleased.connect(self.slider_released)
        self.slider.setEnabled(False) # Disabled until video loaded
        left_panel.addWidget(self.slider)
        
        controls_layout = QHBoxLayout()
        self.step_back_btn = QPushButton("<< Frame")
        self.step_back_btn.clicked.connect(lambda: self.step_frame(-1))
        controls_layout.addWidget(self.step_back_btn)

        self.play_btn = QPushButton("Play")
        self.play_btn.clicked.connect(self.toggle_play)
        self.play_btn.setFixedWidth(100)
        controls_layout.addWidget(self.play_btn)

        self.step_fwd_btn = QPushButton("Frame >>")
        self.step_fwd_btn.clicked.connect(lambda: self.step_frame(1))
        controls_layout.addWidget(self.step_fwd_btn)
        left_panel.addLayout(controls_layout)
        
        self.time_label = QLabel("Video: 0.00s | Exp: 0.00s")
        self.time_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.time_label.setStyleSheet("font-weight: bold; font-size: 14px; margin: 5px;")
        left_panel.addWidget(self.time_label)

        # --- 3. SYNC SETTINGS ---
        sync_group = QGroupBox("Synchronization")
        sync_layout = QHBoxLayout()
        
        self.vid_input = QLineEdit()
        self.vid_input.setPlaceholderText("0.000")
        self.vid_input.setText("0.000")
        self.vid_input.returnPressed.connect(self.apply_sync) 
        sync_layout.addWidget(QLabel("Video T (s):"))
        sync_layout.addWidget(self.vid_input)

        self.cv_input = QLineEdit()
        self.cv_input.setPlaceholderText("0.000")
        self.cv_input.setText("0.000")
        self.cv_input.returnPressed.connect(self.apply_sync)
        sync_layout.addWidget(QLabel("Exp T (s):"))
        sync_layout.addWidget(self.cv_input)
        
        self.apply_btn = QPushButton("Apply Sync")
        self.apply_btn.clicked.connect(self.apply_sync)
        sync_layout.addWidget(self.apply_btn)
        
        sync_group.setLayout(sync_layout)
        left_panel.addWidget(sync_group)
        
        main_layout.addLayout(left_panel, stretch=4)

        # ==================== RIGHT SIDE (Graph) ====================
        right_panel = QVBoxLayout()
        
        # --- Toolbar ---
        toolbar_layout = QHBoxLayout()
        toolbar_layout.setAlignment(Qt.AlignmentFlag.AlignLeft)
        
        toolbar_layout.addWidget(QLabel("<b>Plot Mode:</b>"))
        self.combo_mode = QComboBox()
        self.combo_mode.addItem("Current vs Voltage (CV)")
        self.combo_mode.addItem("Current vs Time (Constant V)")
        self.combo_mode.currentIndexChanged.connect(self.change_plot_mode)
        toolbar_layout.addWidget(self.combo_mode)
        
        toolbar_layout.addSpacing(20)
        
        self.btn_pan = QPushButton("Pan / Drag")
        self.btn_pan.setCheckable(True)
        self.btn_pan.setChecked(True)
        self.btn_pan.clicked.connect(self.set_pan_mode)
        toolbar_layout.addWidget(self.btn_pan)

        self.btn_rect = QPushButton("Rect Zoom")
        self.btn_rect.setCheckable(True)
        self.btn_rect.clicked.connect(self.set_rect_mode)
        toolbar_layout.addWidget(self.btn_rect)

        self.btn_reset = QPushButton("Reset View")
        self.btn_reset.clicked.connect(self.reset_plot_view)
        toolbar_layout.addWidget(self.btn_reset)
        
        right_panel.addLayout(toolbar_layout)

        # --- Plot Widget ---
        pg.setConfigOption('background', 'w')
        pg.setConfigOption('foreground', 'k')
        pg.setConfigOption('antialias', False) 
        
        self.plot_widget = pg.PlotWidget(title="Experiment Data")
        self.plot_widget.setLabel('left', 'Current', units='A')
        self.plot_widget.setLabel('bottom', 'Voltage', units='V')
        self.plot_widget.showGrid(x=True, y=True)
        self.plot_widget.setClipToView(False) 
        
        # Initialize Curves (Empty for now)
        # Background traces (Faint)
        self.bg_fwd = self.plot_widget.plot(pen=pg.mkPen(color=(180, 220, 180), width=1))
        self.bg_rev = self.plot_widget.plot(pen=pg.mkPen(color=(220, 200, 180), width=1))
        
        # Active traces (Strong)
        self.active_fwd = self.plot_widget.plot(pen=pg.mkPen(color='#00AA00', width=2))
        self.active_rev = self.plot_widget.plot(pen=pg.mkPen(color='#FF8800', width=2))
        
        # Current Point
        self.current_point_marker = self.plot_widget.plot(symbol='o', symbolBrush='r', symbolSize=10)
        
        # Legend
        self.legend = self.plot_widget.addLegend()
        self.legend.addItem(self.active_fwd, "Phase 1 (Fwd)")
        self.legend.addItem(self.active_rev, "Phase 2 (Rev)")

        right_panel.addWidget(self.plot_widget)
        main_layout.addLayout(right_panel, stretch=5)

    # ==================== DATA LOADING ====================
    def load_csv_dialog(self):
        fname, _ = QFileDialog.getOpenFileName(self, 'Open CSV', '.', 'CSV Files (*.csv)')
        if fname:
            self.load_csv(fname)

    def load_csv(self, filepath):
        try:
            # Load and Preprocess
            data_tr = pd.read_csv(filepath)
            # Ensure columns exist (adjust indices if your format changes)
            df_cv = data_tr.iloc[:, [3, 5, 12]].copy()
            df_cv.columns = ['Voltage (V)', 'Current (A)', 'Delta Time (s)']
            df_cv['Cumulative Time (s)'] = df_cv['Delta Time (s)'].cumsum()
            
            # Clean
            cols = ['Voltage (V)', 'Current (A)', 'Cumulative Time (s)']
            for c in cols:
                df_cv[c] = pd.to_numeric(df_cv[c], errors='coerce')
            df_cv = df_cv.dropna(subset=cols)
            
            if df_cv.empty:
                raise ValueError("CSV is empty or columns not found.")

            self.df = df_cv
            self.lbl_csv_name.setText(os.path.basename(filepath))
            
            # Extract Arrays
            self.raw_times = self.df['Cumulative Time (s)'].values
            self.raw_voltage = self.df['Voltage (V)'].values
            self.raw_current = self.df['Current (A)'].values
            
            # Determine Split Point (Peak Voltage)
            self.split_idx = np.argmax(self.raw_voltage)
            
            # Refresh Plot
            self.change_plot_mode(self.combo_mode.currentIndex())
            
            # Reset Sync
            self.time_offset = 0.0
            self.cv_input.setText("0.000")
            
        except Exception as e:
            QMessageBox.critical(self, "Error Loading CSV", str(e))

    def load_video_dialog(self):
        fname, _ = QFileDialog.getOpenFileName(self, 'Open Video', '.', 'Video Files (*.mp4 *.avi *.mov)')
        if fname:
            self.load_video(fname)

    def load_video(self, filepath):
        self.video_path = filepath
        self.cap = cv2.VideoCapture(self.video_path)
        
        if not self.cap.isOpened():
            QMessageBox.critical(self, "Error", f"Could not open video: {filepath}")
            return
            
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.video_duration = self.total_frames / self.fps if self.fps > 0 else 0
        self.ms_per_frame = (1000.0 / self.fps) if self.fps > 0 else 33.33
        self.timer_interval = int(self.ms_per_frame)
        
        self.lbl_vid_name.setText(os.path.basename(filepath))
        
        # Enable controls
        self.slider.setEnabled(True)
        self.slider.setRange(0, int(self.video_duration * 1000))
        self.slider.setValue(0)
        
        # Reset Sync
        self.vid_input.setText("0.000")
        self.update_display(0)

    # ==================== PLOT MODES ====================
    def change_plot_mode(self, index):
        if self.df is None:
            return

        # 0 = Current vs Voltage (CV)
        # 1 = Current vs Time (Constant V)
        is_cv_mode = (index == 0)

        # 1. Determine X-Axis Data
        if is_cv_mode:
            x_data = self.raw_voltage
            self.plot_widget.setLabel('bottom', 'Voltage', units='V')
            self.plot_widget.setTitle("Cyclic Voltammetry (I vs V)")
        else:
            x_data = self.raw_times
            self.plot_widget.setLabel('bottom', 'Time', units='s')
            self.plot_widget.setTitle("Current vs Time")

        y_data = self.raw_current

        # 2. Update Background Traces (Static)
        # Fwd Phase (0 to split)
        self.bg_fwd.setData(x_data[:self.split_idx], y_data[:self.split_idx])
        # Rev Phase (split to end)
        self.bg_rev.setData(x_data[self.split_idx:], y_data[self.split_idx:])

        # 3. Calculate View Bounds
        x_min, x_max = np.min(x_data), np.max(x_data)
        y_min, y_max = np.min(y_data), np.max(y_data)
        
        x_pad = (x_max - x_min) * 0.05 if (x_max - x_min) > 0 else 0.1
        y_pad = (y_max - y_min) * 0.05 if (y_max - y_min) > 0 else 0.1
        
        self.view_bounds_x = (x_min - x_pad, x_max + x_pad)
        self.view_bounds_y = (y_min - y_pad, y_max + y_pad)
        
        self.reset_plot_view()
        
        # 4. Trigger display update to draw Active Traces correctly
        self.update_display(self.slider.value())

    # ==================== CORE LOGIC ====================
    def update_display(self, time_ms):
        vid_time_sec = time_ms / 1000.0
        
        # --- A. Update Video ---
        if self.cap and self.cap.isOpened():
            self.cap.set(cv2.CAP_PROP_POS_MSEC, time_ms)
            ret, frame = self.cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                h, w, ch = frame.shape
                bytes_per_line = ch * w
                qt_image = QImage(frame.data, w, h, bytes_per_line, QImage.Format.Format_RGB888)
                pixmap = QPixmap.fromImage(qt_image).scaled(
                    self.video_label.size(), 
                    Qt.AspectRatioMode.KeepAspectRatio, 
                    Qt.TransformationMode.SmoothTransformation
                )
                self.video_label.setPixmap(pixmap)
        
        # --- B. Update Plot ---
        if self.df is None:
            return

        target_exp_time = vid_time_sec + self.time_offset
        self.time_label.setText(f"Video: {vid_time_sec:.2f}s | Exp: {target_exp_time:.2f}s")

        if target_exp_time < 0:
            self.active_fwd.clear()
            self.active_rev.clear()
            self.current_point_marker.clear()
        else:
            # 1. Find current index based on TIME (always syncs by time)
            idx = np.searchsorted(self.raw_times, target_exp_time)
            
            # 2. Determine X-Axis Data based on current Mode
            is_cv_mode = (self.combo_mode.currentIndex() == 0)
            x_data = self.raw_voltage if is_cv_mode else self.raw_times
            y_data = self.raw_current

            # 3. Draw Curves (Split Logic)
            if idx <= self.split_idx:
                # In First Phase
                self.active_fwd.setData(x_data[:idx], y_data[:idx])
                self.active_rev.clear()
            else:
                # In Second Phase
                self.active_fwd.setData(x_data[:self.split_idx], y_data[:self.split_idx])
                self.active_rev.setData(x_data[self.split_idx:idx], y_data[self.split_idx:idx])

            # 4. Draw Red Dot Marker
            if idx < len(x_data):
                self.current_point_marker.setData([x_data[idx]], [y_data[idx]])

    # ==================== CONTROLS & EVENTS ====================
    def apply_sync(self):
        if self.df is None: return
        current_video_time_s = self.slider.value() / 1000.0
        
        try: vid_t = float(self.vid_input.text())
        except ValueError: vid_t = current_video_time_s
        
        try: exp_t = float(self.cv_input.text())
        except ValueError: exp_t = 0.0

        self.time_offset = exp_t - vid_t
        self.vid_input.setText(f"{vid_t:.3f}")
        self.cv_input.setText(f"{exp_t:.3f}")
        self.update_display(self.slider.value())
        self.vid_input.clearFocus()
        self.cv_input.clearFocus()

    def toggle_play(self):
        if self.video_path is None: return
        if self.is_playing:
            self.timer.stop()
            self.play_btn.setText("Play")
        else:
            self.timer.start(self.timer_interval)
            self.play_btn.setText("Pause")
        self.is_playing = not self.is_playing

    def step_frame(self, direction):
        if self.video_path is None: return
        if self.is_playing: self.toggle_play()
        
        current_ms = self.slider.value()
        new_ms = current_ms + (direction * self.ms_per_frame)
        new_ms = max(0, min(new_ms, self.slider.maximum()))
        self.slider.setValue(int(new_ms))
        self.update_display(int(new_ms))

    def next_frame(self):
        current_slider_val = self.slider.value()
        next_val = current_slider_val + self.timer_interval
        if next_val >= self.slider.maximum():
            self.timer.stop()
            self.is_playing = False
            self.play_btn.setText("Play")
        else:
            self.slider.setValue(next_val)
            self.update_display(next_val)

    def slider_moved(self, val):
        self.update_display(val)
        if not self.vid_input.hasFocus():
            self.vid_input.setText(f"{val / 1000.0:.3f}")

    def slider_pressed(self):
        if self.is_playing: self.timer.stop()

    def slider_released(self):
        if self.is_playing: self.timer.start(self.timer_interval)

    def set_pan_mode(self):
        self.btn_pan.setChecked(True)
        self.btn_rect.setChecked(False)
        self.plot_widget.getViewBox().setMouseMode(pg.ViewBox.PanMode)

    def set_rect_mode(self):
        self.btn_pan.setChecked(False)
        self.btn_rect.setChecked(True)
        self.plot_widget.getViewBox().setMouseMode(pg.ViewBox.RectMode)

    def reset_plot_view(self):
        if hasattr(self, 'view_bounds_x'):
            self.plot_widget.setRange(xRange=self.view_bounds_x, yRange=self.view_bounds_y)

    def closeEvent(self, event):
        self.timer.stop()
        if self.cap and self.cap.isOpened():
            self.cap.release()
        event.accept()

# Helper class for sidebar layout
class QFormLayout_Custom(QVBoxLayout):
    def addRow(self, widget1, widget2):
        h = QHBoxLayout()
        h.addWidget(widget1)
        h.addWidget(widget2)
        self.addLayout(h)

# ==================== MAIN ====================
if __name__ == "__main__":
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    
    window = ExperimentVisualizer()
    window.show()
    app.exec()

# Noise Analysis

In [2]:
import sys
import os
import datetime
import pandas as pd
import numpy as np
from scipy.signal import spectrogram
from PyQt6.QtWidgets import (QApplication, QMainWindow, QVBoxLayout, QHBoxLayout, 
                             QWidget, QPushButton, QLabel, QComboBox, QFileDialog, 
                             QFormLayout, QSpinBox, QDoubleSpinBox, QGroupBox, QMessageBox,
                             QDialog, QDialogButtonBox)
from PyQt6.QtCore import Qt, QRectF
import pyqtgraph as pg

# =============================================================================
#  0. INFRASTRUCTURE: DATA CONTEXT
# =============================================================================
class DataContext:
    def __init__(self, df, time_col, signal_col, aux_col=None, aux_label=None, signal_label="Signal", original_filename=""):
        self.df = df
        self.time_col = time_col
        self.signal_col = signal_col
        self.aux_col = aux_col       
        self.aux_label = aux_label   
        self.signal_label = signal_label
        self.original_filename = original_filename 

    def get_time(self): return self.df[self.time_col].values
    def get_signal(self): return self.df[self.signal_col].values
    def get_aux(self): return self.df[self.aux_col].values if self.aux_col else None
    
    def subset(self, t_min, t_max):
        mask = (self.df[self.time_col] >= t_min) & (self.df[self.time_col] <= t_max)
        return DataContext(self.df.loc[mask].copy(), self.time_col, self.signal_col, 
                           self.aux_col, self.aux_label, self.signal_label, self.original_filename)
    
    @property
    def has_aux(self): return self.aux_col is not None

# =============================================================================
#  1. HELPER: CUSTOM AXIS
# =============================================================================
class AuxLookupAxis(pg.AxisItem):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.time_data = np.array([])
        self.aux_data = np.array([])

    def set_lookup_data(self, time_arr, aux_arr):
        self.time_data = time_arr
        self.aux_data = aux_arr

    def get_val_at_time(self, t):
        if len(self.time_data) == 0: return np.nan
        return np.interp(t, self.time_data, self.aux_data, left=np.nan, right=np.nan)

    def tickStrings(self, values, scale, spacing):
        if len(self.time_data) == 0: return []
        try:
            mapped = np.interp(values, self.time_data, self.aux_data, left=np.nan, right=np.nan)
            return ["" if np.isnan(v) else f"{v:.3f}" for v in mapped]
        except: return [""] * len(values)

# =============================================================================
#  2. COLUMN SELECTION DIALOG
# =============================================================================
class ColumnSelectionDialog(QDialog):
    def __init__(self, columns, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Select Columns")
        layout = QVBoxLayout(self)
        form = QFormLayout()
        self.combo_time = QComboBox(); self.combo_time.addItems(columns)
        self.combo_signal = QComboBox(); self.combo_signal.addItems(columns)
        form.addRow("Time Column (X):", self.combo_time)
        form.addRow("Signal Column (Y):", self.combo_signal)
        layout.addLayout(form)
        btns = QDialogButtonBox(QDialogButtonBox.StandardButton.Ok | QDialogButtonBox.StandardButton.Cancel)
        btns.accepted.connect(self.accept)
        btns.rejected.connect(self.reject)
        layout.addWidget(btns)

    def get_selection(self):
        return self.combo_time.currentText(), self.combo_signal.currentText()

# =============================================================================
#  3. ABSTRACT STRATEGY INTERFACE
# =============================================================================
class AnalysisStrategy:
    def name(self): raise NotImplementedError
    def description(self): raise NotImplementedError
    def load_data(self, parent_widget, filepath=None): raise NotImplementedError 
    def get_required_parameters(self): return {}
    def execute(self, context, params): raise NotImplementedError
    def plot_results(self, raw_plot_widget, result_layout_widget, results): raise NotImplementedError
    def save_results(self, results, parent_widget): raise NotImplementedError
    
    def clear_overlays(self, raw_plot_widget): pass

    def _get_save_path(self, parent_widget, suffix, ctx=None):
        folder = QFileDialog.getExistingDirectory(parent_widget, "Select Output Folder")
        if not folder: return None
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        
        name_part = ""
        if ctx and ctx.original_filename:
            base_name = os.path.splitext(os.path.basename(ctx.original_filename))[0]
            name_part = f"_{base_name}"
            
        return f"{folder}/Analysis_{ts}_{suffix}{name_part}.csv"

    def _save_raw_slice(self, parent_widget, ctx, time_lbl="Time", sig_lbl="Signal", aux_lbl="Aux"):
        """Helper to save the raw sliced data with customizable headers"""
        path = self._get_save_path(parent_widget, "RawSlice", ctx)
        if path:
            data = {
                time_lbl: ctx.get_time(),
                sig_lbl: ctx.get_signal()
            }
            if ctx.has_aux:
                data[aux_lbl] = ctx.get_aux()
                
            df = pd.DataFrame(data)
            df.to_csv(path, index=False)

# =============================================================================
#  4. CONCRETE STRATEGIES
# =============================================================================

# --- A. CV Specific Strategy ---
class CV_PolyFFT(AnalysisStrategy):
    def __init__(self):
        self.trend_item = None

    def name(self): return "CV: Poly Detrend + PSD"
    def description(self): return "Specific to CV data (Voltage/Current). Includes Voltage Axis."

    def load_data(self, parent, filepath=None):
        if not filepath:
            filepath, _ = QFileDialog.getOpenFileName(parent, 'Open CV CSV', '.', 'CSV Files (*.csv)')
        if not filepath: return None, None
        
        try:
            df = pd.read_csv(filepath, dtype=str)
            df = df.iloc[:, [3, 5, 12]].copy()
            df.columns = ['Voltage', 'Current', 'DeltaTime']
            for c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
            df = df.dropna()
            df['Time'] = df['DeltaTime'].cumsum()
            
            ctx = DataContext(df, 'Time', 'Current', 'Voltage', 'Voltage (V)', 'Current (A)', filepath)
            return ctx, filepath
        except Exception as e:
            QMessageBox.warning(parent, "Load Error", f"CV Load Failed: {e}")
            return None, None

    def get_required_parameters(self):
        return {
            'Poly Order': {'type': 'int', 'default': 3, 'min': 1, 'max': 10},
            'Sampling Rate (Hz)': {'type': 'float', 'default': 1000.0}
        }

    def execute(self, ctx, params):
        t = ctx.get_time()
        y = ctx.get_signal()
        t_c = t - t[0]
        coeffs = np.polyfit(t_c, y, params['Poly Order'])
        trend = np.polyval(coeffs, t_c)
        resid = y - trend
        
        n = len(resid)
        win = np.hanning(n)
        fft_res = np.fft.fft(resid * win)
        freqs = np.fft.fftfreq(n, d=1/params['Sampling Rate (Hz)'])
        mask = freqs > 0
        power = (np.abs(fft_res[mask]) ** 2) / n
        
        return {
            'ctx': ctx, 'trend': trend, 'resid': resid,
            'freqs': freqs[mask], 'power': power
        }

    def clear_overlays(self, raw_plot):
        if self.trend_item and self.trend_item in raw_plot.listDataItems():
            raw_plot.removeItem(self.trend_item)
            self.trend_item = None

    def plot_results(self, raw_plot, res_layout, res):
        self.clear_overlays(raw_plot)
        self.trend_item = raw_plot.plot(res['ctx'].get_time(), res['trend'], pen=pg.mkPen('r', width=2), name="Fit")
        
        res_layout.clear()
        ctx = res['ctx']
        ax_args = {}
        if ctx.has_aux:
            ax = AuxLookupAxis(orientation='top')
            ax.set_lookup_data(ctx.get_time(), ctx.get_aux())
            ax.setLabel(ctx.aux_label)
            ax_args = {'axisItems': {'top': ax}}

        p1 = res_layout.addPlot(row=0, col=0, title="Residuals", **ax_args)
        p1.plot(ctx.get_time(), res['resid'], pen='g')
        p1.setLabel('left', 'Delta ' + ctx.signal_label)
        p1.showGrid(x=True, y=True)
        if ctx.has_aux: p1.showAxis('top')

        res_layout.nextRow()
        p2 = res_layout.addPlot(row=1, col=0, title="Power Spectral Density")
        p2.plot(res['freqs'], res['power'], pen='b')
        p2.setLogMode(x=True, y=True)
        p2.setLabel('bottom', 'Frequency', 'Hz')
        p2.showGrid(x=True, y=True)

    def save_results(self, res, parent):
        # 1. Save Raw Slice with Specific Headers
        self._save_raw_slice(parent, res['ctx'], time_lbl="Time (s)", sig_lbl="Current (A)", aux_lbl="Voltage (V)")

        # 2. Save Residuals & Trend
        path_res = self._get_save_path(parent, "CV_Results", res['ctx'])
        if path_res:
            data = {
                'Time (s)': res['ctx'].get_time(), 
                'Current (A)': res['ctx'].get_signal(), 
                'Trend (A)': res['trend'], 
                'Residual (A)': res['resid']
            }
            if res['ctx'].has_aux: 
                data['Voltage (V)'] = res['ctx'].get_aux()
                
            df = pd.DataFrame(data)
            df.to_csv(path_res, index=False)
        
        # 3. Save PSD
        path_psd = self._get_save_path(parent, "CV_PSD", res['ctx'])
        if path_psd:
            pd.DataFrame({'Frequency (Hz)': res['freqs'], 'Power Density': res['power']}).to_csv(path_psd, index=False)

        QMessageBox.information(parent, "Saved", f"Data saved successfully.")

# --- B. CV STFT Strategy ---
class CV_STFT(CV_PolyFFT):
    def name(self): return "CV: STFT Spectrogram"
    
    def get_required_parameters(self):
        p = super().get_required_parameters()
        p.update({
            'Window Size': {'type': 'int', 'default': 256},
            'Overlap Ratio': {'type': 'float', 'default': 0.8}
        })
        return p

    def execute(self, ctx, params):
        parent_res = super().execute(ctx, params)
        resid = parent_res['resid']
        fs = params['Sampling Rate (Hz)']
        nperseg = params['Window Size']
        noverlap = int(nperseg * params['Overlap Ratio'])
        
        f, t_spec, Sxx = spectrogram(resid, fs, nperseg=nperseg, noverlap=noverlap)
        return {
            'ctx': ctx, 'trend': parent_res['trend'], 'resid': resid,
            'spec_t': t_spec + ctx.get_time()[0], 'spec_f': f, 'spec_p': Sxx
        }

    def plot_results(self, raw_plot, res_layout, res):
        self.clear_overlays(raw_plot)
        self.trend_item = raw_plot.plot(res['ctx'].get_time(), res['trend'], pen=pg.mkPen('r', width=2, style=Qt.PenStyle.DashLine))
        
        res_layout.clear()
        ctx = res['ctx']
        ax_args = {}
        if ctx.has_aux:
            ax = AuxLookupAxis(orientation='top')
            ax.set_lookup_data(ctx.get_time(), ctx.get_aux())
            ax.setLabel(ctx.aux_label)
            ax_args = {'axisItems': {'top': ax}}
            
        p1 = res_layout.addPlot(row=0, col=0, title="Noise Trace", **ax_args)
        p1.plot(ctx.get_time(), res['resid'], pen='g')
        if ctx.has_aux: p1.showAxis('top')
        
        res_layout.nextRow()
        # Spectrogram Plot
        p2 = res_layout.addPlot(row=1, col=0, title="Spectrogram")
        p1.setXLink(p2)
        
        img = pg.ImageItem()
        p2.addItem(img)
        power_db = 10 * np.log10(res['spec_p'] + 1e-12)
        img.setImage(power_db.T)
        
        t, f = res['spec_t'], res['spec_f']
        img.setRect(QRectF(t[0], f[0], t[-1]-t[0], f[-1]-f[0]))
        p2.setLabel('left', 'Frequency', 'Hz')
        p2.setLabel('bottom', 'Time', 's')

        # Add Histogram / Color Bar
        hist = pg.HistogramLUTItem()
        hist.setImageItem(img)
        hist.gradient.loadPreset('viridis')
        res_layout.addItem(hist, row=1, col=1)

    def save_results(self, res, parent):
        # 1. Save Raw Slice
        self._save_raw_slice(parent, res['ctx'], time_lbl="Time (s)", sig_lbl="Current (A)", aux_lbl="Voltage (V)")

        # 2. Save Residuals Trace
        path_res = self._get_save_path(parent, "STFT_Residuals", res['ctx'])
        if path_res:
             data = {
                 'Time (s)': res['ctx'].get_time(), 
                 'Trend (A)': res['trend'], 
                 'Residual (A)': res['resid']
             }
             if res['ctx'].has_aux:
                 data['Voltage (V)'] = res['ctx'].get_aux()
             
             df = pd.DataFrame(data)
             df.to_csv(path_res, index=False)

        QMessageBox.information(parent, "Saved", f"Data saved successfully.")

# --- C. GENERIC Strategy ---
class Generic_PolyFFT(AnalysisStrategy):
    def __init__(self):
        self.trend_item = None

    def name(self): return "Generic: Poly Detrend + PSD"
    def description(self): return "For any CSV. You select Time and Signal columns."

    def load_data(self, parent, filepath=None):
        if not filepath:
            filepath, _ = QFileDialog.getOpenFileName(parent, 'Open Generic CSV', '.', 'CSV Files (*.csv)')
        if not filepath: return None, None
        
        try:
            df = pd.read_csv(filepath)
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if len(numeric_cols) < 2: raise ValueError("CSV needs at least 2 numeric columns.")
            
            dlg = ColumnSelectionDialog(numeric_cols, parent)
            if dlg.exec() == QDialog.DialogCode.Accepted:
                t_col, y_col = dlg.get_selection()
                df = df.sort_values(by=t_col)
                ctx = DataContext(df, t_col, y_col, None, None, y_col, filepath)
                return ctx, filepath
            else:
                return None, None
        except Exception as e:
            QMessageBox.warning(parent, "Load Error", str(e))
            return None, None

    def get_required_parameters(self):
        return {
            'Poly Order': {'type': 'int', 'default': 2},
            'Sampling Rate (Hz)': {'type': 'float', 'default': 1.0} 
        }

    def execute(self, ctx, params):
        t = ctx.get_time()
        y = ctx.get_signal()
        t_c = t - t[0]
        coeffs = np.polyfit(t_c, y, params['Poly Order'])
        trend = np.polyval(coeffs, t_c)
        resid = y - trend
        
        n = len(resid)
        win = np.hanning(n)
        fft_res = np.fft.fft(resid * win)
        freqs = np.fft.fftfreq(n, d=1/params['Sampling Rate (Hz)'])
        mask = freqs > 0
        power = (np.abs(fft_res[mask]) ** 2) / n
        
        return {
            'ctx': ctx, 'trend': trend, 'resid': resid,
            'freqs': freqs[mask], 'power': power
        }

    def clear_overlays(self, raw_plot):
        if self.trend_item and self.trend_item in raw_plot.listDataItems():
            raw_plot.removeItem(self.trend_item)
            self.trend_item = None

    def plot_results(self, raw_plot, res_layout, res):
        self.clear_overlays(raw_plot)
        self.trend_item = raw_plot.plot(res['ctx'].get_time(), res['trend'], pen=pg.mkPen('r', width=2), name="Fit")
        
        res_layout.clear()
        p1 = res_layout.addPlot(row=0, col=0, title="Residuals")
        p1.plot(res['ctx'].get_time(), res['resid'], pen='g')
        p1.showGrid(x=True, y=True)
        
        res_layout.nextRow()
        p2 = res_layout.addPlot(row=1, col=0, title="Power Spectral Density")
        p2.plot(res['freqs'], res['power'], pen='b')
        p2.setLogMode(x=True, y=True)
        p2.showGrid(x=True, y=True)

    def save_results(self, res, parent):
        self._save_raw_slice(parent, res['ctx'], time_lbl="Time", sig_lbl="Signal", aux_lbl="Aux")
        
        path = self._get_save_path(parent, "Generic_Results", res['ctx'])
        if path:
            pd.DataFrame({'Time': res['ctx'].get_time(), 'Signal': res['ctx'].get_signal(), 'Trend': res['trend'], 'Residual': res['resid']}).to_csv(path, index=False)
        
        QMessageBox.information(parent, "Saved", f"Data saved successfully.")


# =============================================================================
#  5. MAIN APP
# =============================================================================
class UniversalAnalyzer(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Universal Time-Series Noise Analyzer")
        self.resize(1400, 950)
        pg.setConfigOption('background', 'w')
        pg.setConfigOption('foreground', 'k')
        
        self.strategies = [CV_PolyFFT(), CV_STFT(), Generic_PolyFFT()]
        self.curr_strat = self.strategies[0]
        self.data_ctx = None
        self.results = None
        self.last_loaded_file = None # Persist filename
        
        self.init_ui()

    def init_ui(self):
        w = QWidget(); self.setCentralWidget(w)
        layout = QHBoxLayout(w)
        
        side = QVBoxLayout()
        side_widget = QWidget(); side_widget.setFixedWidth(300); side_widget.setLayout(side)
        
        grp_strat = QGroupBox("1. Analysis Mode")
        l_strat = QVBoxLayout()
        self.combo_strat = QComboBox()
        for s in self.strategies: self.combo_strat.addItem(s.name())
        self.combo_strat.currentIndexChanged.connect(self.on_strat_change)
        self.lbl_desc = QLabel(self.curr_strat.description())
        self.lbl_desc.setWordWrap(True)
        l_strat.addWidget(self.combo_strat)
        l_strat.addWidget(self.lbl_desc)
        grp_strat.setLayout(l_strat); side.addWidget(grp_strat)
        
        grp_load = QGroupBox("2. Data")
        l_load = QVBoxLayout()
        self.btn_load = QPushButton("Load Data")
        self.btn_load.clicked.connect(lambda: self.load_data_flow(None))
        self.lbl_file = QLabel("No Data")
        l_load.addWidget(self.btn_load); l_load.addWidget(self.lbl_file)
        grp_load.setLayout(l_load); side.addWidget(grp_load)
        
        grp_param = QGroupBox("3. Parameters")
        self.l_param = QFormLayout()
        grp_param.setLayout(self.l_param)
        side.addWidget(grp_param)
        self.param_inputs = {}
        
        grp_act = QGroupBox("4. Execute")
        l_act = QVBoxLayout()
        self.btn_run = QPushButton("Run Analysis")
        self.btn_run.clicked.connect(self.run_analysis)
        self.btn_save = QPushButton("Save Results")
        self.btn_save.clicked.connect(self.save_results)
        self.btn_save.setEnabled(False)
        l_act.addWidget(self.btn_run); l_act.addWidget(self.btn_save)
        grp_act.setLayout(l_act); side.addWidget(grp_act)
        
        side.addStretch()
        layout.addWidget(side_widget)
        
        center = QVBoxLayout()
        self.lbl_hover = QLabel("Hover for Info")
        self.lbl_hover.setAlignment(Qt.AlignmentFlag.AlignCenter)
        center.addWidget(self.lbl_hover)
        
        self.ax_top = AuxLookupAxis(orientation='top')
        self.plot_raw = pg.PlotWidget(axisItems={'top': self.ax_top}, title="Raw Data (Select Region)")
        self.plot_raw.showGrid(x=True, y=True)
        self.region = pg.LinearRegionItem()
        self.region.setZValue(10)
        self.plot_raw.scene().sigMouseMoved.connect(self.on_mouse_move)
        
        center.addWidget(self.plot_raw, 1)
        self.plot_layout = pg.GraphicsLayoutWidget()
        center.addWidget(self.plot_layout, 2)
        layout.addLayout(center)
        self.refresh_params()

    def on_strat_change(self, idx):
        # Clear overlays from previous strategy before switching
        self.curr_strat.clear_overlays(self.plot_raw)
        
        self.curr_strat = self.strategies[idx]
        self.lbl_desc.setText(self.curr_strat.description())
        self.refresh_params()
        self.plot_layout.clear()
        
        # Try to auto-reload
        if self.last_loaded_file:
            self.lbl_file.setText("Attempting reload...")
            success = self.load_data_flow(self.last_loaded_file)
            if not success:
                 self.data_ctx = None
                 self.plot_raw.clear()
                 self.lbl_file.setText("Reload Failed. Pick Data.")

    def refresh_params(self):
        while self.l_param.count():
            item = self.l_param.takeAt(0)
            if item.widget(): item.widget().deleteLater()
        self.param_inputs = {}
        
        for k, v in self.curr_strat.get_required_parameters().items():
            if v['type'] == 'int':
                w = QSpinBox(); w.setRange(v.get('min', 1), v.get('max', 99999)); w.setValue(v['default'])
            elif v['type'] == 'float':
                w = QDoubleSpinBox(); w.setRange(v.get('min', 0.0), v.get('max', 1e9)); w.setValue(v['default']); w.setSingleStep(0.1)
            self.param_inputs[k] = w
            self.l_param.addRow(k, w)

    def load_data_flow(self, filepath=None):
        ctx, loaded_path = self.curr_strat.load_data(self, filepath)
        if ctx:
            self.data_ctx = ctx
            self.last_loaded_file = loaded_path
            self.lbl_file.setText(f"Loaded: {len(ctx.df)} rows")
            
            # Setup Raw Plot
            self.plot_raw.clear()
            self.plot_raw.addItem(self.region)
            self.plot_raw.plot(ctx.get_time(), ctx.get_signal(), pen='k')
            self.plot_raw.setLabel('bottom', "Time")
            self.plot_raw.setLabel('left', ctx.signal_label)
            
            if ctx.has_aux:
                self.ax_top.set_lookup_data(ctx.get_time(), ctx.get_aux())
                self.ax_top.setLabel(ctx.aux_label)
                self.plot_raw.showAxis('top')
            else:
                self.plot_raw.hideAxis('top')
                self.ax_top.set_lookup_data([], [])

            # Smart Region
            t = ctx.get_time()
            if len(t) > 1:
                mid = (t[-1] + t[0]) / 2
                span = (t[-1] - t[0]) * 0.2
                self.region.setRegion([mid - span, mid + span])
                
                # --- SMART SAMPLING RATE CALCULATION ---
                # Calculate mean delta_t
                dt_vals = np.diff(t)
                dt_avg = np.mean(dt_vals)
                if dt_avg > 0:
                    fs_est = 1.0 / dt_avg
                    if 'Sampling Rate (Hz)' in self.param_inputs:
                        self.param_inputs['Sampling Rate (Hz)'].setValue(fs_est)

            return True
        return False

    def run_analysis(self):
        if not self.data_ctx: return
        min_t, max_t = self.region.getRegion()
        sub_ctx = self.data_ctx.subset(min_t, max_t)
        if len(sub_ctx.get_time()) < 10:
            QMessageBox.warning(self, "Error", "Selection too small")
            return
            
        p = {k: w.value() for k, w in self.param_inputs.items()}
        try:
            self.results = self.curr_strat.execute(sub_ctx, p)
            self.curr_strat.plot_results(self.plot_raw, self.plot_layout, self.results)
            self.btn_save.setEnabled(True)
        except Exception as e:
            QMessageBox.critical(self, "Analysis Failed", str(e))

    def save_results(self):
        if self.results: self.curr_strat.save_results(self.results, self)

    def on_mouse_move(self, pos):
        if not self.data_ctx: return
        vb = self.plot_raw.plotItem.vb
        if self.plot_raw.sceneBoundingRect().contains(pos):
            pt = vb.mapSceneToView(pos)
            t = pt.x()
            msg = f"Time: {t:.3f} s"
            if self.data_ctx.has_aux:
                val = self.ax_top.get_val_at_time(t)
                if not np.isnan(val): msg += f" | {self.data_ctx.aux_label}: {val:.3f}"
            self.lbl_hover.setText(msg)

if __name__ == "__main__":
    app = QApplication.instance()
    if not app: app = QApplication(sys.argv)
    w = UniversalAnalyzer()
    w.show()
    app.exec()

# Video Tracking

In [2]:
import sys
import os
import cv2
import numpy as np
import pandas as pd
from PyQt6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, 
                             QLabel, QPushButton, QGroupBox, QFileDialog, QProgressBar, 
                             QMessageBox, QSlider, QStyle)
from PyQt6.QtCore import Qt, QThread, pyqtSignal, QPointF
from PyQt6.QtGui import QImage, QPixmap, QPainter, QPen, QColor, QPolygonF
import pyqtgraph as pg

# =============================================================================
#  CUSTOM WIDGET: ROI DRAWING LAYER
# =============================================================================
class VideoLabel(QLabel):
    """
    Handles video display and ROI drawing. 
    Uses Normalized Coordinates (0.0-1.0) to ensure scaling robustness.
    """
    roi_finalized = pyqtSignal(list) # Emits list of normalized (x, y) tuples

    def __init__(self):
        super().__init__()
        self.setMouseTracking(True)
        self.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.setStyleSheet("background-color: #000; border: 1px solid #333;")
        self.setMinimumSize(480, 360)
        
        self.drawing_active = False
        self.norm_points = [] # Points stored as 0.0-1.0
        self.current_pixmap = None

    def start_drawing(self):
        self.norm_points = []
        self.drawing_active = True
        self.setCursor(Qt.CursorShape.CrossCursor)
        self.update()

    def reset_roi(self):
        self.norm_points = []
        self.drawing_active = False
        self.roi_finalized.emit([])
        self.update()

    def mousePressEvent(self, event):
        if not self.drawing_active or not self.current_pixmap: return
        
        # 1. Calculate the rectangle where the image is actually drawn
        # (QLabel centers the image and keeps aspect ratio)
        lbl_w, lbl_h = self.width(), self.height()
        img_w, img_h = self.current_pixmap.width(), self.current_pixmap.height()
        
        scale = min(lbl_w / img_w, lbl_h / img_h)
        actual_w = int(img_w * scale)
        actual_h = int(img_h * scale)
        
        offset_x = (lbl_w - actual_w) // 2
        offset_y = (lbl_h - actual_h) // 2
        
        # 2. Normalize mouse click relative to that rectangle
        click_x = event.pos().x() - offset_x
        click_y = event.pos().y() - offset_y
        
        # 3. Check bounds and save
        if 0 <= click_x <= actual_w and 0 <= click_y <= actual_h:
            nx = click_x / actual_w
            ny = click_y / actual_h
            
            if event.button() == Qt.MouseButton.LeftButton:
                self.norm_points.append((nx, ny))
                self.update()
            elif event.button() == Qt.MouseButton.RightButton:
                self.drawing_active = False
                self.setCursor(Qt.CursorShape.ArrowCursor)
                self.roi_finalized.emit(self.norm_points)
                self.update()

    def paintEvent(self, event):
        super().paintEvent(event) # Draw the pixmap
        
        if not self.norm_points or not self.current_pixmap: return
        
        # Re-calculate geometry to draw the polygon overlay correctly
        lbl_w, lbl_h = self.width(), self.height()
        img_w, img_h = self.current_pixmap.width(), self.current_pixmap.height()
        scale = min(lbl_w / img_w, lbl_h / img_h)
        actual_w = int(img_w * scale)
        actual_h = int(img_h * scale)
        offset_x = (lbl_w - actual_w) // 2
        offset_y = (lbl_h - actual_h) // 2

        painter = QPainter(self)
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)
        pen = QPen(QColor(0, 255, 255), 2)
        painter.setPen(pen)
        
        # Convert normalized points to screen points
        screen_points = []
        for nx, ny in self.norm_points:
            px = offset_x + (nx * actual_w)
            py = offset_y + (ny * actual_h)
            screen_points.append(QPointF(px, py))
            
        poly = QPolygonF(screen_points)
        painter.drawPolyline(poly)
        
        # Connect last to first if finished
        if not self.drawing_active and len(screen_points) > 2:
            painter.drawLine(screen_points[-1], screen_points[0])
            painter.setBrush(QColor(0, 255, 255, 40))
            painter.drawPolygon(poly)

# =============================================================================
#  WORKER: DATA EXPORT (BLOCKING PROCESS)
# =============================================================================
class ExportThread(QThread):
    progress = pyqtSignal(int)
    finished = pyqtSignal()
    
    def __init__(self, video_path, roi_points):
        super().__init__()
        self.video_path = video_path
        self.roi_points = roi_points # Normalized
        self.running = True

    def run(self):
        cap = cv2.VideoCapture(self.video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        prev_gray = None
        data_log = []
        
        # Pre-calculate ROI mask if points exist
        roi_mask = None
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        if len(self.roi_points) > 2:
            roi_mask = np.zeros((h, w), dtype=np.uint8)
            pts = np.array([[int(nx*w), int(ny*h)] for nx, ny in self.roi_points], dtype=np.int32)
            cv2.fillPoly(roi_mask, [pts], 255)

        idx = 0
        while self.running:
            ret, frame = cap.read()
            if not ret: break
            
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
            # Metrics defaults
            kinetic_energy = 0.0
            vert_flux = 0.0
            lat_instability = 0.0
            
            if prev_gray is not None:
                # Farneback Optical Flow
                flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                
                # Apply ROI Mask to flow
                if roi_mask is not None:
                    # Bitwise AND to zero out flow outside mask
                    # flow is (H, W, 2). Mask is (H, W).
                    # We need to broadcast mask
                    mask_bool = roi_mask > 0
                    flow_active = flow[mask_bool] # Returns flattened array of only active pixels (N, 2)
                else:
                    flow_active = flow.reshape(-1, 2)
                
                if flow_active.shape[0] > 0:
                    fx = flow_active[:, 0]
                    fy = flow_active[:, 1]
                    
                    # 1. Kinetic Energy (Mean Magnitude)
                    mag = np.sqrt(fx**2 + fy**2)
                    kinetic_energy = np.mean(mag)
                    
                    # 2. Vertical Flux (Mean Y velocity)
                    # Note: In OpenCV, Y increases downwards. So Up is Negative.
                    # Let's invert it so Up is Positive for graph intuitiveness.
                    vert_flux = np.mean(-fy) 
                    
                    # 3. Lateral Instability (Std Dev of X velocity)
                    lat_instability = np.std(fx)

            prev_gray = gray
            
            data_log.append({
                'Time (s)': idx / fps,
                'Kinetic Energy': kinetic_energy,
                'Vertical Flux': vert_flux,
                'Lateral Instability': lat_instability
            })
            
            if idx % 10 == 0:
                self.progress.emit(idx)
            idx += 1
            
        cap.release()
        
        # Save CSV
        df = pd.DataFrame(data_log)
        base, _ = os.path.splitext(self.video_path)
        df.to_csv(f"{base}_flow_metrics.csv", index=False)
        self.finished.emit()

    def stop(self):
        self.running = False

# =============================================================================
#  MAIN APPLICATION
# =============================================================================
class FlowAnalysisApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Optical Flow Kinetics Analyzer")
        self.resize(1400, 900)
        
        # State
        self.video_path = None
        self.cap = None
        self.timer = None
        self.total_frames = 0
        self.fps = 30
        self.roi_points = [] # Normalized
        self.prev_gray = None # For live flow viz
        
        self.init_ui()
        
    def init_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        layout = QHBoxLayout(central)
        
        # --- LEFT: CONTROLS ---
        left = QVBoxLayout()
        left_cont = QWidget(); left_cont.setFixedWidth(300); left_cont.setLayout(left)
        layout.addWidget(left_cont)
        
        # Input
        grp_in = QGroupBox("1. Input")
        l_in = QVBoxLayout()
        btn_load = QPushButton("Load Video")
        btn_load.clicked.connect(self.load_video)
        self.lbl_path = QLabel("No Video")
        l_in.addWidget(btn_load); l_in.addWidget(self.lbl_path)
        grp_in.setLayout(l_in); left.addWidget(grp_in)
        
        # ROI
        grp_roi = QGroupBox("2. Region of Interest")
        l_roi = QVBoxLayout()
        self.btn_draw = QPushButton("Draw Polygon")
        self.btn_draw.setCheckable(True)
        self.btn_draw.clicked.connect(self.toggle_drawing)
        self.btn_reset = QPushButton("Reset ROI")
        self.btn_reset.clicked.connect(self.reset_roi)
        l_roi.addWidget(self.btn_draw); l_roi.addWidget(self.btn_reset)
        l_roi.addWidget(QLabel("Left Click: Add Point\nRight Click: Close Shape"))
        grp_roi.setLayout(l_roi); left.addWidget(grp_roi)
        
        # Playback
        grp_play = QGroupBox("3. Preview")
        l_play = QVBoxLayout()
        self.slider = QSlider(Qt.Orientation.Horizontal)
        self.slider.sliderMoved.connect(self.seek_frame)
        self.btn_play = QPushButton("Play / Pause")
        self.btn_play.clicked.connect(self.toggle_play)
        l_play.addWidget(self.slider); l_play.addWidget(self.btn_play)
        grp_play.setLayout(l_play); left.addWidget(grp_play)
        
        # Export
        grp_ex = QGroupBox("4. Analysis")
        l_ex = QVBoxLayout()
        self.btn_export = QPushButton("Run Full Analysis (CSV)")
        self.btn_export.setStyleSheet("background-color: #d4f1f4; font-weight: bold;")
        self.btn_export.clicked.connect(self.start_export)
        self.pbar = QProgressBar()
        l_ex.addWidget(self.btn_export); l_ex.addWidget(self.pbar)
        grp_ex.setLayout(l_ex); left.addWidget(grp_ex)
        
        left.addStretch()

        # --- MIDDLE: VIDEO ---
        mid = QVBoxLayout()
        layout.addLayout(mid)
        
        self.lbl_video = VideoLabel()
        self.lbl_video.roi_finalized.connect(self.on_roi_finalized)
        
        self.lbl_flow = QLabel("Optical Flow Viz")
        self.lbl_flow.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.lbl_flow.setStyleSheet("background-color: #000; border: 1px solid #333;")
        self.lbl_flow.setMinimumSize(480, 360)
        
        mid.addWidget(QLabel("<b>Raw Video (Draw ROI Here)</b>"))
        mid.addWidget(self.lbl_video)
        mid.addWidget(QLabel("<b>Real-time Flow Visualization</b>"))
        mid.addWidget(self.lbl_flow)

        # --- RIGHT: HYPOTHESIS GRAPHS ---
        right = QVBoxLayout()
        right_cont = QWidget(); right_cont.setFixedWidth(400); right_cont.setLayout(right)
        layout.addWidget(right_cont)
        
        pg.setConfigOption('background', 'w')
        pg.setConfigOption('foreground', 'k')
        
        # Graph 1
        self.plot_energy = pg.PlotWidget(title="1. Kinetic Energy (Mass Transport)")
        self.plot_energy.setLabel('left', 'Mean Mag')
        self.plot_energy.setLabel('bottom', 'Frame')
        self.curve_energy = self.plot_energy.plot(pen='r')
        right.addWidget(self.plot_energy)
        
        # Graph 2
        self.plot_vert = pg.PlotWidget(title="2. Vertical Flux (Buoyancy)")
        self.plot_vert.setLabel('left', 'Mean Y-Vel (Up+)')
        self.curve_vert = self.plot_vert.plot(pen='g')
        right.addWidget(self.plot_vert)
        
        # Graph 3
        self.plot_lat = pg.PlotWidget(title="3. Lateral Instability (Rupture)")
        self.plot_lat.setLabel('left', 'StdDev X-Vel')
        self.curve_lat = self.plot_lat.plot(pen='b')
        right.addWidget(self.plot_lat)

        # Graph Data Storage (for preview only)
        self.graph_data = {'frames': [], 'energy': [], 'vert': [], 'lat': []}

    # ================= LOGIC =================

    def load_video(self):
        path, _ = QFileDialog.getOpenFileName(self, "Open Video", "", "Video (*.mp4 *.avi)")
        if path:
            self.video_path = path
            self.lbl_path.setText(os.path.basename(path))
            self.cap = cv2.VideoCapture(path)
            self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
            self.fps = self.cap.get(cv2.CAP_PROP_FPS)
            self.slider.setRange(0, self.total_frames)
            self.slider.setValue(0)
            self.prev_gray = None
            self.graph_data = {'frames': [], 'energy': [], 'vert': [], 'lat': []}
            self.seek_frame(0)

    def toggle_drawing(self):
        if self.btn_draw.isChecked():
            self.lbl_video.start_drawing()
            self.btn_draw.setText("Right Click Video to Finish")
        else:
            self.btn_draw.setText("Draw Polygon")

    def reset_roi(self):
        self.lbl_video.reset_roi()
        self.roi_points = []
        self.prev_gray = None # Reset flow history
        self.seek_frame(self.slider.value())

    def on_roi_finalized(self, points):
        self.roi_points = points
        self.btn_draw.setChecked(False)
        self.btn_draw.setText("Draw Polygon")
        self.prev_gray = None
        self.seek_frame(self.slider.value())

    def toggle_play(self):
        if not self.timer:
            self.timer = self.startTimer(int(1000/self.fps)) # ~33ms
        else:
            self.killTimer(self.timer)
            self.timer = None

    def timerEvent(self, event):
        # Frame Advance logic
        curr = self.slider.value()
        if curr < self.total_frames:
            self.seek_frame(curr + 1)
            self.slider.setValue(curr + 1)
        else:
            self.killTimer(self.timer)
            self.timer = None

    def seek_frame(self, frame_idx):
        if not self.cap: return
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = self.cap.read()
        if not ret: return

        # 1. Update Raw Video View
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb.shape
        qimg = QImage(rgb.data, w, h, ch*w, QImage.Format.Format_RGB888)
        self.lbl_video.current_pixmap = QPixmap.fromImage(qimg)
        self.lbl_video.setPixmap(self.lbl_video.current_pixmap.scaled(
            self.lbl_video.size(), Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation))

        # 2. Update Flow View (Live Preview)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Prepare Mask from ROI if it exists
        roi_mask = None
        if len(self.roi_points) > 2:
            roi_mask = np.zeros((h, w), dtype=np.uint8)
            pts = np.array([[int(nx*w), int(ny*h)] for nx, ny in self.roi_points], dtype=np.int32)
            cv2.fillPoly(roi_mask, [pts], 255)

        if self.prev_gray is not None:
            # Downscale for preview performance (Live Only)
            prev_s = cv2.resize(self.prev_gray, (0,0), fx=0.5, fy=0.5)
            curr_s = cv2.resize(gray, (0,0), fx=0.5, fy=0.5)
            
            flow = cv2.calcOpticalFlowFarneback(prev_s, curr_s, None, 0.5, 3, 15, 3, 5, 1.2, 0)
            
            # Visualization (HSV)
            mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            hsv = np.zeros((flow.shape[0], flow.shape[1], 3), dtype=np.uint8)
            hsv[..., 1] = 255
            hsv[..., 0] = ang * 180 / np.pi / 2
            hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
            
            # Apply Mask to Visuals (Resize mask to match preview scale)
            if roi_mask is not None:
                mask_s = cv2.resize(roi_mask, (hsv.shape[1], hsv.shape[0]), interpolation=cv2.INTER_NEAREST)
                hsv = cv2.bitwise_and(hsv, hsv, mask=mask_s)

            flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
            fh, fw, fch = flow_rgb.shape
            fqimg = QImage(flow_rgb.data, fw, fh, fch*fw, QImage.Format.Format_RGB888)
            self.lbl_flow.setPixmap(QPixmap.fromImage(fqimg).scaled(
                self.lbl_flow.size(), Qt.AspectRatioMode.KeepAspectRatio))
            
            # Calculate Preview Metrics (Approximate)
            if roi_mask is not None:
                mask_bool = mask_s > 0
                flow_active = flow[mask_bool]
            else:
                flow_active = flow.reshape(-1, 2)
            
            if flow_active.size > 0:
                self.update_live_graphs(frame_idx, flow_active)

        self.prev_gray = gray

    def update_live_graphs(self, frame, flow_vecs):
        fx = flow_vecs[:, 0]
        fy = flow_vecs[:, 1]
        
        k_e = np.mean(np.sqrt(fx**2 + fy**2))
        v_f = np.mean(-fy) # Invert Y
        l_i = np.std(fx)
        
        self.graph_data['frames'].append(frame)
        self.graph_data['energy'].append(k_e)
        self.graph_data['vert'].append(v_f)
        self.graph_data['lat'].append(l_i)
        
        # Keep only last 100 points for performance
        if len(self.graph_data['frames']) > 100:
            for k in self.graph_data: self.graph_data[k].pop(0)
            
        self.curve_energy.setData(self.graph_data['frames'], self.graph_data['energy'])
        self.curve_vert.setData(self.graph_data['frames'], self.graph_data['vert'])
        self.curve_lat.setData(self.graph_data['frames'], self.graph_data['lat'])

    def start_export(self):
        if not self.video_path: return
        self.btn_export.setEnabled(False)
        self.btn_export.setText("Processing...")
        if self.timer: self.toggle_play() # Pause preview
        
        self.exporter = ExportThread(self.video_path, self.roi_points)
        self.exporter.progress.connect(self.pbar.setValue)
        self.exporter.finished.connect(self.on_export_finished)
        self.pbar.setMaximum(self.total_frames)
        self.exporter.start()

    def on_export_finished(self):
        self.btn_export.setEnabled(True)
        self.btn_export.setText("Run Full Analysis (CSV)")
        QMessageBox.information(self, "Done", "Analysis complete. CSV saved next to video.")

    def closeEvent(self, e):
        if self.timer: self.killTimer(self.timer)
        if hasattr(self, 'exporter') and self.exporter.isRunning():
            self.exporter.stop()
        e.accept()

if __name__ == "__main__":
    app = QApplication.instance()
    if app is None: app = QApplication(sys.argv)
    window = FlowAnalysisApp()
    window.show()
    app.exec()

# Bubble Masking (Deprecated)

In [2]:
import sys
import os
import cv2
import numpy as np
import pandas as pd
from PyQt6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, 
                             QLabel, QPushButton, QGroupBox, QFileDialog, QFormLayout, 
                             QDoubleSpinBox, QSpinBox, QProgressBar, QMessageBox, QTabWidget,
                             QSlider, QStyle, QCheckBox)
from PyQt6.QtCore import Qt, QThread, pyqtSignal, QMutex, QWaitCondition
from PyQt6.QtGui import QImage, QPixmap
import pyqtgraph as pg

# =============================================================================
#  WORKER THREAD: HEAVY PROCESSING
# =============================================================================
class VideoProcessor(QThread):
    # Signals
    frame_processed = pyqtSignal(object, object, object, float, dict) 
    finished_export = pyqtSignal()
    progress_update = pyqtSignal(int)
    background_learned = pyqtSignal(object) # Emit the learned background image
    
    def __init__(self):
        super().__init__()
        self.video_path = None
        self.params = {}
        
        # State control
        self.running = False
        self.paused = False
        self.export_mode = False
        self.current_frame_idx = 0
        self.target_seek_frame = -1
        
        # Thread synchronization
        self.mutex = QMutex()
        self.condition = QWaitCondition()

        # CV Objects
        self.cap = None
        self.static_bg_gray = None # The learned background
        self.fps = 30.0
        self.total_frames = 100

    def load_video(self, path):
        self.video_path = path
        temp_cap = cv2.VideoCapture(path)
        if temp_cap.isOpened():
            self.fps = temp_cap.get(cv2.CAP_PROP_FPS)
            self.total_frames = int(temp_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        temp_cap.release()
        self.static_bg_gray = None # Reset background on new load

    def learn_background(self, start_t, end_t, downscale_factor):
        """Blocking function to learn background from a specific time range."""
        if not self.video_path: return

        cap = cv2.VideoCapture(self.video_path)
        start_f = int(start_t * self.fps)
        end_f = int(end_t * self.fps)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
        
        frames = []
        count = 0
        max_frames_to_read = end_f - start_f
        
        # Read frames
        while count < max_frames_to_read:
            ret, frame = cap.read()
            if not ret: break
            
            # Apply downscale immediately to save memory
            if downscale_factor != 1.0:
                h, w = frame.shape[:2]
                frame = cv2.resize(frame, (int(w * downscale_factor), int(h * downscale_factor)))
            
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frames.append(gray)
            count += 1
            
        cap.release()
        
        if frames:
            # Calculate Median (Robust against noise)
            # Stack frames into a 3D array (Frame, H, W) and take median along axis 0
            stack = np.stack(frames, axis=0)
            median_frame = np.median(stack, axis=0).astype(np.uint8)
            
            self.mutex.lock()
            self.static_bg_gray = median_frame
            self.mutex.unlock()
            
            self.background_learned.emit(self.static_bg_gray)

    def seek(self, frame_idx):
        self.mutex.lock()
        self.target_seek_frame = frame_idx
        if self.paused:
            self.condition.wakeAll()
        self.mutex.unlock()

    def set_paused(self, is_paused):
        self.mutex.lock()
        self.paused = is_paused
        if not is_paused:
            self.condition.wakeAll()
        self.mutex.unlock()

    def run(self):
        if not self.video_path: return
        
        self.cap = cv2.VideoCapture(self.video_path)
        
        prev_gray = None
        data_log = [] 
        self.running = True
        
        while self.running:
            self.mutex.lock()
            
            # Handle Pausing
            while self.paused and self.target_seek_frame == -1:
                self.condition.wait(self.mutex)
                if not self.running: 
                    self.mutex.unlock()
                    return

            # Handle Seeking
            if self.target_seek_frame != -1:
                self.cap.set(cv2.CAP_PROP_POS_FRAMES, self.target_seek_frame)
                self.current_frame_idx = self.target_seek_frame
                self.target_seek_frame = -1
                prev_gray = None 
                
            self.mutex.unlock()

            # Read Frame
            ret, frame = self.cap.read()
            if not ret: 
                if self.export_mode: break
                self.set_paused(True)
                continue

            self.current_frame_idx = int(self.cap.get(cv2.CAP_PROP_POS_FRAMES))

            # ---------------------------
            # CV PIPELINE
            # ---------------------------
            scale = self.params.get('downscale', 0.5)
            if scale != 1.0:
                width = int(frame.shape[1] * scale)
                height = int(frame.shape[0] * scale)
                frame = cv2.resize(frame, (width, height))
            
            timestamp = self.current_frame_idx / self.fps if self.fps > 0 else 0
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
            # --- 1. Segmentation (Static Background) ---
            gas_area = 0
            fg_mask = np.zeros_like(gray)
            
            if self.static_bg_gray is not None:
                if self.static_bg_gray.shape != gray.shape:
                    # Resize bg if params changed mid-run
                    self.static_bg_gray = cv2.resize(self.static_bg_gray, (gray.shape[1], gray.shape[0]))

                # A. Absolute Difference
                diff = cv2.absdiff(gray, self.static_bg_gray)
                
                # B. Blur (Softens noise)
                blur_k = self.params.get('blur', 5)
                if blur_k > 0:
                    # Ensure odd kernel
                    k = blur_k if blur_k % 2 == 1 else blur_k + 1
                    diff = cv2.GaussianBlur(diff, (k, k), 0)

                # C. Threshold
                thresh_val = self.params.get('threshold', 30)
                _, fg_mask = cv2.threshold(diff, thresh_val, 255, cv2.THRESH_BINARY)
                
                # D. Morphology (Refinement)
                morph_k = self.params.get('morph_size', 3)
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_k, morph_k))
                
                # Open: Removes small dots (noise)
                if self.params.get('do_open', True):
                    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
                
                # Close: Fills internal holes (makes bubbles solid)
                if self.params.get('do_close', True):
                    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel)

                gas_area = cv2.countNonZero(fg_mask)

            # --- 2. Optical Flow ---
            flow_energy = 0.0
            flow_vis = np.zeros_like(frame) 
            
            if prev_gray is not None:
                flow = cv2.calcOpticalFlowFarneback(
                    prev_gray, gray, None,
                    pyr_scale=0.5, levels=3, winsize=15,
                    iterations=3, poly_n=5, poly_sigma=1.2, flags=0
                )
                mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
                flow_energy = np.mean(mag)
                
                if not self.export_mode:
                    hsv = np.zeros_like(frame)
                    hsv[..., 1] = 255
                    hsv[..., 0] = ang * 180 / np.pi / 2
                    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
                    flow_vis = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

            prev_gray = gray
            
            # Output
            metrics = {
                'frame': self.current_frame_idx,
                'time': timestamp,
                'gas_area': gas_area,
                'flow_energy': flow_energy
            }
            
            if self.export_mode:
                data_log.append(metrics)
                if self.current_frame_idx % 10 == 0:
                    self.progress_update.emit(self.current_frame_idx)
            else:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mask_rgb = cv2.cvtColor(fg_mask, cv2.COLOR_GRAY2RGB)
                flow_rgb = cv2.cvtColor(flow_vis, cv2.COLOR_BGR2RGB)
                
                self.frame_processed.emit(frame_rgb, mask_rgb, flow_rgb, timestamp, metrics)
                self.msleep(30) 
            
        self.cap.release()
        
        if self.export_mode:
            df = pd.DataFrame(data_log)
            save_path = self.video_path + "_tracking.csv"
            df.to_csv(save_path, index=False)
            self.finished_export.emit()

    def stop(self):
        self.running = False
        self.condition.wakeAll()
        self.wait()


# =============================================================================
#  MAIN GUI
# =============================================================================
class FeatureExtractorApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Hydrodynamic Feature Extractor (CV Prediction Prep)")
        self.resize(1600, 950)
        
        self.processor = VideoProcessor()
        self.processor.frame_processed.connect(self.update_ui)
        self.processor.finished_export.connect(self.export_finished)
        self.processor.progress_update.connect(self.update_progress)
        self.processor.background_learned.connect(self.on_bg_learned)
        
        self.scrub_cap = None 
        self.is_scrubbing = False
        self.live_data = {'time': [], 'gas_area': [], 'flow_energy': []}
        
        self.init_ui()

    def init_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        layout = QHBoxLayout(central)
        
        # --- LEFT: CONTROLS ---
        left_container = QWidget()
        left_container.setFixedWidth(340) # Slightly wider for new controls
        left_panel = QVBoxLayout(left_container)
        
        # 1. File
        grp_file = QGroupBox("1. Input")
        lyt_file = QVBoxLayout()
        self.btn_load = QPushButton("Load Video")
        self.btn_load.clicked.connect(self.load_video)
        self.lbl_vid = QLabel("No Video")
        lyt_file.addWidget(self.btn_load)
        lyt_file.addWidget(self.lbl_vid)
        
        # General Params
        lyt_file.addWidget(QLabel("Global Downscale:"))
        self.spin_downscale = QDoubleSpinBox()
        self.spin_downscale.setRange(0.1, 1.0)
        self.spin_downscale.setValue(0.5)
        self.spin_downscale.setSingleStep(0.1)
        lyt_file.addWidget(self.spin_downscale)
        
        grp_file.setLayout(lyt_file)
        left_panel.addWidget(grp_file)

        # 2. Background Learning (Static)
        grp_bg = QGroupBox("2. Static Background")
        lyt_bg = QFormLayout()
        
        self.spin_bg_start = QDoubleSpinBox()
        self.spin_bg_start.setRange(0, 9999)
        self.spin_bg_start.setValue(0.0)
        lyt_bg.addRow("Start (s):", self.spin_bg_start)
        
        self.spin_bg_end = QDoubleSpinBox()
        self.spin_bg_end.setRange(0, 9999)
        self.spin_bg_end.setValue(1.0)
        lyt_bg.addRow("End (s):", self.spin_bg_end)
        
        self.btn_learn_bg = QPushButton("Learn Background")
        self.btn_learn_bg.clicked.connect(self.learn_background)
        self.btn_learn_bg.setEnabled(False)
        lyt_bg.addRow(self.btn_learn_bg)
        
        self.lbl_bg_status = QLabel("Not Learned")
        self.lbl_bg_status.setStyleSheet("color: red")
        lyt_bg.addRow("Status:", self.lbl_bg_status)
        
        grp_bg.setLayout(lyt_bg)
        left_panel.addWidget(grp_bg)
        
        # 3. Mask Refinement
        grp_mask = QGroupBox("3. Mask Refinement")
        lyt_mask = QFormLayout()
        
        self.spin_blur = QSpinBox()
        self.spin_blur.setRange(0, 50)
        self.spin_blur.setValue(5)
        lyt_mask.addRow("Blur (Softness):", self.spin_blur)
        
        self.spin_thresh = QSpinBox() 
        self.spin_thresh.setRange(1, 255)
        self.spin_thresh.setValue(30)
        lyt_mask.addRow("Threshold (Sens):", self.spin_thresh)

        self.spin_morph_size = QSpinBox()
        self.spin_morph_size.setRange(1, 30)
        self.spin_morph_size.setValue(3)
        lyt_mask.addRow("Morph Size:", self.spin_morph_size)

        self.chk_open = QCheckBox("Remove Speckles (Open)")
        self.chk_open.setChecked(True)
        lyt_mask.addRow(self.chk_open)
        
        self.chk_close = QCheckBox("Fill Holes (Close)")
        self.chk_close.setChecked(True)
        lyt_mask.addRow(self.chk_close)
        
        grp_mask.setLayout(lyt_mask)
        left_panel.addWidget(grp_mask)
        
        # 4. Playback Controls
        grp_play = QGroupBox("4. Playback")
        lyt_play = QVBoxLayout()
        
        self.slider = QSlider(Qt.Orientation.Horizontal)
        self.slider.setRange(0, 100)
        self.slider.setEnabled(False)
        self.slider.sliderPressed.connect(self.on_slider_press)
        self.slider.sliderMoved.connect(self.on_slider_move)
        self.slider.sliderReleased.connect(self.on_slider_release)
        lyt_play.addWidget(self.slider)

        self.btn_play = QPushButton()
        self.update_play_button(False)
        self.btn_play.clicked.connect(self.toggle_play)
        self.btn_play.setEnabled(False)
        lyt_play.addWidget(self.btn_play)
        
        grp_play.setLayout(lyt_play)
        left_panel.addWidget(grp_play)
        
        # 5. Export
        self.btn_export = QPushButton("Process Full Video & Save CSV")
        self.btn_export.setStyleSheet("background-color: #ffcccc; font-weight: bold;")
        self.btn_export.clicked.connect(self.start_export)
        left_panel.addWidget(self.btn_export)
        
        self.pbar = QProgressBar()
        left_panel.addWidget(self.pbar)
        
        left_panel.addStretch()
        layout.addWidget(left_container)
        
        # --- MIDDLE & RIGHT (Visualization) ---
        mid_panel = QVBoxLayout()
        
        self.tabs = QTabWidget()
        tab_composite = QWidget()
        grid = QVBoxLayout(tab_composite)
        
        row1 = QHBoxLayout()
        self.lbl_orig = QLabel("Original")
        self.lbl_orig.setScaledContents(True)
        self.lbl_orig.setFixedSize(400, 300)
        self.lbl_orig.setStyleSheet("background: #000; border: 1px solid #333;")
        
        self.lbl_mask = QLabel("Gas Mask (Area)")
        self.lbl_mask.setScaledContents(True)
        self.lbl_mask.setFixedSize(400, 300)
        self.lbl_mask.setStyleSheet("background: #000; border: 1px solid #333;")
        
        row1.addWidget(self.lbl_orig)
        row1.addWidget(self.lbl_mask)
        
        row2 = QHBoxLayout()
        self.lbl_flow = QLabel("Optical Flow (Energy)")
        self.lbl_flow.setScaledContents(True)
        self.lbl_flow.setFixedSize(400, 300)
        self.lbl_flow.setStyleSheet("background: #000; border: 1px solid #333;")
        
        row2.addWidget(self.lbl_flow)
        
        # Add Background Preview
        self.lbl_bg_preview = QLabel("Background Ref")
        self.lbl_bg_preview.setScaledContents(True)
        self.lbl_bg_preview.setFixedSize(400, 300)
        self.lbl_bg_preview.setStyleSheet("background: #222; border: 1px dashed #666;")
        row2.addWidget(self.lbl_bg_preview)
        
        grid.addLayout(row1)
        grid.addLayout(row2)
        
        self.tabs.addTab(tab_composite, "Live View")
        mid_panel.addWidget(self.tabs)
        layout.addLayout(mid_panel)
        
        # --- RIGHT: GRAPHS ---
        right_panel = QVBoxLayout()
        pg.setConfigOption('background', 'w')
        pg.setConfigOption('foreground', 'k')
        
        self.plot_area = pg.PlotWidget(title="Gas Area (Segmentation)")
        self.plot_area.setLabel('left', 'Pixels')
        self.plot_area.setLabel('bottom', 'Time', 's')
        self.curve_area = self.plot_area.plot(pen='r')
        right_panel.addWidget(self.plot_area)
        
        self.plot_flow = pg.PlotWidget(title="Kinetic Energy (Optical Flow)")
        self.plot_flow.setLabel('left', 'Magnitude')
        self.plot_flow.setLabel('bottom', 'Time', 's')
        self.curve_flow = self.plot_flow.plot(pen='b')
        right_panel.addWidget(self.plot_flow)
        
        layout.addLayout(right_panel)

    # =========================================================================
    #  LOGIC
    # =========================================================================
    def load_video(self):
        path, _ = QFileDialog.getOpenFileName(self, "Open Video", "", "Video Files (*.mp4 *.avi)")
        if path:
            self.processor.load_video(path)
            self.scrub_cap = cv2.VideoCapture(path)
            self.lbl_vid.setText(os.path.basename(path))
            self.pbar.setMaximum(self.processor.total_frames)
            
            # Reset UI
            self.slider.setRange(0, self.processor.total_frames)
            self.slider.setValue(0)
            self.slider.setEnabled(True)
            self.btn_play.setEnabled(True)
            self.btn_learn_bg.setEnabled(True)
            self.update_play_button(False)
            self.lbl_bg_status.setText("Not Learned")
            self.lbl_bg_status.setStyleSheet("color: red")
            self.lbl_bg_preview.clear()
            self.lbl_bg_preview.setText("Background Ref")
            
            # Start Processor Thread (Paused)
            self.processor.params = self.get_params()
            self.processor.paused = True
            if not self.processor.isRunning():
                self.processor.start()
            
            self.show_static_frame(0)

    def get_params(self):
        return {
            'downscale': self.spin_downscale.value(),
            'threshold': self.spin_thresh.value(),
            'blur': self.spin_blur.value(),
            'morph_size': self.spin_morph_size.value(),
            'do_open': self.chk_open.isChecked(),
            'do_close': self.chk_close.isChecked()
        }

    def learn_background(self):
        self.btn_learn_bg.setText("Learning...")
        self.btn_learn_bg.setEnabled(False)
        QApplication.processEvents() # Force UI update
        
        st = self.spin_bg_start.value()
        et = self.spin_bg_end.value()
        ds = self.spin_downscale.value()
        
        # Call the blocking function in the thread (it's safe here as long as we aren't playing)
        # Ideally this should be async, but for a 1 second range it's fast enough.
        self.processor.learn_background(st, et, ds)
        
        self.btn_learn_bg.setText("Learn Background")
        self.btn_learn_bg.setEnabled(True)

    def on_bg_learned(self, bg_img):
        self.lbl_bg_status.setText("Learned!")
        self.lbl_bg_status.setStyleSheet("color: green")
        
        # Display the learned background
        h, w = bg_img.shape
        qimg = QImage(bg_img.data, w, h, w, QImage.Format.Format_Grayscale8)
        pixmap = QPixmap.fromImage(qimg)
        self.lbl_bg_preview.setPixmap(pixmap)
        
        # If we are currently viewing a frame, force update to apply mask immediately
        if self.processor.paused:
            # We trigger a seek to current frame to force a re-process in the thread
            # This allows the user to see the mask update immediately after learning BG
            self.processor.seek(self.slider.value())

    def update_play_button(self, is_playing):
        icon = self.style().standardIcon(QStyle.StandardPixmap.SP_MediaPause if is_playing else QStyle.StandardPixmap.SP_MediaPlay)
        self.btn_play.setIcon(icon)
        self.btn_play.setText(" Pause Processing" if is_playing else " Play & Analyze")

    def toggle_play(self):
        self.processor.params = self.get_params()
        if self.processor.paused:
            self.processor.set_paused(False)
            self.update_play_button(True)
        else:
            self.processor.set_paused(True)
            self.update_play_button(False)

    # --- SLIDER LOGIC ---
    def on_slider_press(self):
        self.is_scrubbing = True
        self.was_playing_before_scrub = not self.processor.paused
        if self.was_playing_before_scrub:
            self.processor.set_paused(True)

    def on_slider_move(self, val):
        self.show_static_frame(val)

    def on_slider_release(self):
        self.is_scrubbing = False
        target_frame = self.slider.value()
        
        # Important: Update params in case user changed them while scrubbing
        self.processor.params = self.get_params()
        
        self.processor.seek(target_frame)
        
        if self.was_playing_before_scrub:
            self.processor.set_paused(False)
        else:
            self.update_play_button(False)

    def show_static_frame(self, frame_idx):
        if self.scrub_cap and self.scrub_cap.isOpened():
            self.scrub_cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = self.scrub_cap.read()
            if ret:
                scale = self.spin_downscale.value()
                h, w = frame.shape[:2]
                new_w, new_h = int(w * scale), int(h * scale)
                frame = cv2.resize(frame, (new_w, new_h))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                self.set_image(self.lbl_orig, frame)
                self.lbl_mask.clear()
                self.lbl_flow.clear()

    # --- UPDATES ---
    def update_ui(self, orig, mask, flow, timestamp, metrics):
        if self.is_scrubbing: return
        
        self.slider.blockSignals(True)
        self.slider.setValue(metrics['frame'])
        self.slider.blockSignals(False)

        self.set_image(self.lbl_orig, orig)
        self.set_image(self.lbl_mask, mask)
        self.set_image(self.lbl_flow, flow)
        
        self.live_data['time'].append(timestamp)
        self.live_data['gas_area'].append(metrics['gas_area'])
        self.live_data['flow_energy'].append(metrics['flow_energy'])
        
        if len(self.live_data['time']) > 300:
            self.live_data['time'].pop(0)
            self.live_data['gas_area'].pop(0)
            self.live_data['flow_energy'].pop(0)
            
        self.curve_area.setData(self.live_data['time'], self.live_data['gas_area'])
        self.curve_flow.setData(self.live_data['time'], self.live_data['flow_energy'])

    def set_image(self, label, img_array):
        if img_array is None: return
        h, w, ch = img_array.shape
        bytes_per_line = ch * w
        qimg = QImage(img_array.data, w, h, bytes_per_line, QImage.Format.Format_RGB888)
        pixmap = QPixmap.fromImage(qimg)
        label.setPixmap(pixmap)

    def start_export(self):
        if self.processor.static_bg_gray is None:
            QMessageBox.warning(self, "Warning", "Please learn background before exporting!")
            return
            
        self.processor.set_paused(True)
        self.processor.params = self.get_params()
        self.processor.export_mode = True
        self.processor.seek(0)
        self.pbar.setValue(0)
        self.btn_export.setEnabled(False)
        self.btn_export.setText("Processing...")
        self.processor.set_paused(False)

    def update_progress(self, frame_num):
        self.pbar.setValue(frame_num)

    def export_finished(self):
        self.processor.set_paused(True)
        self.btn_export.setText("Process Full Video & Save CSV")
        self.btn_export.setEnabled(True)
        QMessageBox.information(self, "Success", "Processing complete.\nCSV saved.")
    
    def closeEvent(self, event):
        if self.processor.isRunning():
            self.processor.stop()
        event.accept()

if __name__ == "__main__":
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    window = FeatureExtractorApp()
    window.show()
    app.exec()

# Single Cycle

In [9]:
import sys
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from PyQt6.QtWidgets import (QApplication, QMainWindow, QVBoxLayout, QHBoxLayout, 
                             QWidget, QPushButton, QLabel, QComboBox, QFileDialog, 
                             QFormLayout, QGroupBox, QMessageBox,
                             QCheckBox, QLineEdit)
from PyQt6.QtCore import Qt
import pyqtgraph as pg

# =============================================================================
#  1. DATA STRUCTURES
# =============================================================================
class CycleSegment:
    """
    Represents a single bubble evolution cycle.
    """
    def __init__(self, raw_t, raw_i, raw_v):
        self.raw_t = raw_t
        self.raw_i = raw_i
        self.raw_v = raw_v
        
        self.avg_voltage = np.mean(raw_v) if len(raw_v) > 0 else 0
        self.duration = raw_t[-1] - raw_t[0] if len(raw_t) > 0 else 0
        
        self.norm_t = np.array([])
        self.norm_i = np.array([])
        self._normalize()

    def _normalize(self):
        if len(self.raw_t) < 2: return
        
        # Normalize Time: 0.0 start, 1.0 end
        t_start = self.raw_t[0]
        t_range = self.raw_t[-1] - t_start
        if t_range == 0: t_range = 1e-9
        self.norm_t = (self.raw_t - t_start) / t_range
        
        # Normalize Current: 0.0 min, 1.0 max
        i_min = np.min(self.raw_i)
        i_range = np.max(self.raw_i) - i_min
        if i_range == 0: i_range = 1e-9
        self.norm_i = (self.raw_i - i_min) / i_range

# =============================================================================
#  2. ANALYSIS STRATEGIES
# =============================================================================
class AnalysisStrategy:
    def name(self): raise NotImplementedError
    def get_params(self): return {}
    def process(self, t, i, v, params): raise NotImplementedError
    def visualize_results(self, plot_widget, cycles, layout_item=None): raise NotImplementedError

class BasePeakStrategy(AnalysisStrategy):
    """
    Base class that handles the common Peak Detection logic.
    Subclasses only need to implement visualize_results.
    """
    def get_params(self):
        return {
            'Invert Signal': {'type': 'bool', 'default': True, 'desc': 'Check if bubble detachment is a valley (dip)'},
            'Prominence': {'type': 'float', 'default': 1e-6, 'desc': 'Vertical threshold (Amps).'},
            'Min Distance': {'type': 'int', 'default': 50, 'desc': 'Min samples between peaks.'},
            'Width': {'type': 'int', 'default': 5, 'desc': 'Min width of peak in samples.'}
        }

    def process(self, t, i, v, params):
        sig = -i if params['Invert Signal'] else i
        
        peaks, _ = find_peaks(sig, 
                              prominence=params['Prominence'], 
                              distance=params['Min Distance'], 
                              width=params['Width'])
        
        cycles = []
        # Need at least 2 peaks to form a cycle.
        # Logic naturally discards pre-1st peak and post-last peak data.
        if len(peaks) > 1:
            for k in range(len(peaks) - 1):
                start_idx = peaks[k]
                end_idx = peaks[k+1]
                
                c_t = t[start_idx : end_idx+1]
                c_i = i[start_idx : end_idx+1]
                c_v = v[start_idx : end_idx+1]
                
                if len(c_t) > 5:
                    cycles.append(CycleSegment(c_t, c_i, c_v))
        
        return cycles, peaks

# --- Mode 1: The Original Overlay ---
class OverlayVisualization(BasePeakStrategy):
    def name(self): return "Mode 1: Normalized Cycle Overlay"
    
    def visualize_results(self, plot_widget, cycles, layout_item):
        plot_widget.clear()
        plot_widget.setTitle(f"Overlay: {len(cycles)} Cycles (Color = Voltage)")
        plot_widget.setLabel('left', 'Norm. Current (0-1)')
        plot_widget.setLabel('bottom', 'Norm. Time (0-1)')
        
        if not cycles: return

        # 1. Setup Color Map
        all_avg_v = [c.avg_voltage for c in cycles]
        min_v = min(all_avg_v)
        max_v = max(all_avg_v)
        if max_v == min_v: max_v += 1e-9
        
        # We need to access the histogram from the layout to update levels
        if layout_item:
            layout_item.setLevels(min_v, max_v)
            pg_cmap = layout_item.gradient.colorMap()
        else:
            return

        # 2. Plot Lines
        for cycle in cycles:
            norm_v_val = (cycle.avg_voltage - min_v) / (max_v - min_v)
            color = pg_cmap.mapToQColor(norm_v_val)
            color.setAlpha(150)
            plot_widget.plot(cycle.norm_t, cycle.norm_i, pen=pg.mkPen(color, width=2))

# --- Mode 2: The New Scatter Plot ---
class ScatterVisualization(BasePeakStrategy):
    def name(self): return "Mode 2: Duration vs Voltage Scatter"
    
    def visualize_results(self, plot_widget, cycles, layout_item):
        plot_widget.clear()
        plot_widget.setTitle(f"Kinetics: {len(cycles)} Cycles Detected")
        plot_widget.setLabel('bottom', 'Average Voltage', 'V')
        plot_widget.setLabel('left', 'Cycle Duration', 's')
        
        if not cycles: return

        # Extract Data
        x_data = [c.avg_voltage for c in cycles] # Voltage
        y_data = [c.duration for c in cycles]    # Time
        
        # Plot Scatter
        scatter = pg.ScatterPlotItem(size=10, pen=pg.mkPen(None), brush=pg.mkBrush(255, 0, 0, 150))
        scatter.addPoints(x=x_data, y=y_data)
        plot_widget.addItem(scatter)
        
        # Disable Histogram if present (it's not used here)
        if layout_item:
            # We can't easily hide it without removing from layout, 
            # so let's just set it to a neutral range or ignore it.
            pass

# =============================================================================
#  3. MAIN APPLICATION
# =============================================================================
class BubbleCycleAnalyzer(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Bubble Cycle Analysis & Normalization")
        self.resize(1400, 1000)
        
        pg.setConfigOption('background', 'w')
        pg.setConfigOption('foreground', 'k')
        pg.setConfigOption('antialias', True)
        
        self.df = None
        # Register Strategies
        self.strategies = [OverlayVisualization(), ScatterVisualization()]
        self.current_strategy = self.strategies[0]
        self.param_inputs = {} 
        
        self.init_ui()

    def init_ui(self):
        w = QWidget(); self.setCentralWidget(w)
        layout = QHBoxLayout(w)
        
        # --- SIDEBAR ---
        side = QVBoxLayout()
        side_widget = QWidget(); side_widget.setFixedWidth(320); side_widget.setLayout(side)
        
        # 1. Load
        grp_load = QGroupBox("1. Data Loading")
        l_load = QVBoxLayout()
        self.btn_load = QPushButton("Load CV CSV")
        self.btn_load.clicked.connect(self.load_data)
        self.lbl_file = QLabel("No File Loaded")
        self.lbl_file.setWordWrap(True)
        l_load.addWidget(self.btn_load); l_load.addWidget(self.lbl_file)
        grp_load.setLayout(l_load); side.addWidget(grp_load)
        
        # 2. Strategy
        grp_strat = QGroupBox("2. Analysis Mode")
        l_strat = QVBoxLayout()
        self.combo_strat = QComboBox()
        for s in self.strategies: self.combo_strat.addItem(s.name())
        self.combo_strat.currentIndexChanged.connect(self.change_strategy)
        l_strat.addWidget(self.combo_strat)
        grp_strat.setLayout(l_strat); side.addWidget(grp_strat)
        
        # 3. Parameters
        self.grp_param = QGroupBox("3. Peak Detection Parameters")
        self.form_param = QFormLayout()
        self.grp_param.setLayout(self.form_param)
        side.addWidget(self.grp_param)
        
        # 4. Action
        grp_act = QGroupBox("4. Process")
        l_act = QVBoxLayout()
        self.btn_process = QPushButton("Run Analysis")
        self.btn_process.setStyleSheet("background-color: #d4f1f4; font-weight: bold; padding: 10px;")
        self.btn_process.clicked.connect(self.run_analysis)
        self.btn_process.setEnabled(False)
        l_act.addWidget(self.btn_process)
        grp_act.setLayout(l_act); side.addWidget(grp_act)
        
        side.addStretch()
        layout.addWidget(side_widget)
        
        # --- PLOT AREA ---
        plot_layout = pg.GraphicsLayoutWidget()
        layout.addWidget(plot_layout)
        
        # Plot 1: Raw Overview
        self.plot_raw = plot_layout.addPlot(row=0, col=0, title="Raw Data (Select Region)")
        self.plot_raw.setLabel('left', 'Current', 'A')
        self.plot_raw.setLabel('bottom', 'Time', 's')
        self.region = pg.LinearRegionItem()
        self.region.setZValue(10)
        self.plot_raw.addItem(self.region)
        self.region.sigRegionChanged.connect(self.on_region_change)
        
        # Plot 2: Detailed Selection & Peaks
        self.plot_detail = plot_layout.addPlot(row=1, col=0, title="Selection & Detected Breakpoints")
        self.plot_detail.setLabel('left', 'Current', 'A')
        self.plot_detail.setLabel('bottom', 'Time', 's')
        self.plot_detail.showGrid(x=True, y=True)
        
        # Plot 3: Result (Dynamic based on strategy)
        self.plot_result = plot_layout.addPlot(row=2, col=0, title="Analysis Result")
        self.plot_result.showGrid(x=True, y=True)
        
        # Color Bar (Helper for Mode 1)
        self.cmap_histogram = pg.HistogramLUTItem()
        self.cmap_histogram.gradient.loadPreset('viridis')
        plot_layout.addItem(self.cmap_histogram, row=2, col=1)

        self.refresh_params_ui()

    # --- LOGIC ---
    def change_strategy(self, idx):
        self.current_strategy = self.strategies[idx]
        self.refresh_params_ui()
        # If we have data loaded, we can re-run automatically or wait for user
        # Let's wait for user to click Run
        self.plot_result.clear()
        self.plot_result.setTitle(f"Analysis Result ({self.current_strategy.name()})")

    def load_data(self):
        fname, _ = QFileDialog.getOpenFileName(self, "Open CSV", ".", "CSV (*.csv)")
        if not fname: return
        
        try:
            df = pd.read_csv(fname, dtype=str)
            df = df.iloc[:, [3, 5, 12]].copy()
            df.columns = ['Voltage', 'Current', 'DeltaTime']
            for c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
            df = df.dropna()
            df['Time'] = df['DeltaTime'].cumsum()
            
            self.df = df
            self.lbl_file.setText(f"Loaded: {len(df)} rows")
            
            self.plot_raw.clear()
            self.plot_raw.addItem(self.region)
            self.plot_raw.plot(df['Time'].values, df['Current'].values, pen='k')
            
            t = df['Time'].values
            mid = (t[-1]+t[0])/2
            span = (t[-1]-t[0])*0.1
            self.region.setRegion([mid-span, mid+span])
            
            self.btn_process.setEnabled(True)
            self.update_detail_view()
            
        except Exception as e:
            QMessageBox.critical(self, "Load Error", str(e))

    def refresh_params_ui(self):
        while self.form_param.count():
            item = self.form_param.takeAt(0)
            if item.widget(): item.widget().deleteLater()
        self.param_inputs = {}
        
        params = self.current_strategy.get_params()
        for key, info in params.items():
            if info['type'] == 'bool':
                w = QCheckBox()
                w.setChecked(info['default'])
            else:
                w = QLineEdit()
                w.setText(str(info['default']))
                if 'desc' in info: w.setToolTip(info['desc'])
            
            self.param_inputs[key] = w
            self.form_param.addRow(key, w)

    def on_region_change(self):
        self.update_detail_view()

    def update_detail_view(self):
        if self.df is None: return
        min_t, max_t = self.region.getRegion()
        mask = (self.df['Time'] >= min_t) & (self.df['Time'] <= max_t)
        sub_df = self.df.loc[mask]
        
        self.plot_detail.clear()
        if sub_df.empty: return
        self.plot_detail.plot(sub_df['Time'].values, sub_df['Current'].values, pen='k')

    def run_analysis(self):
        if self.df is None: return
        
        # 1. Get Data Selection
        min_t, max_t = self.region.getRegion()
        mask = (self.df['Time'] >= min_t) & (self.df['Time'] <= max_t)
        sub_df = self.df.loc[mask]
        if len(sub_df) < 10: 
            QMessageBox.warning(self, "Warning", "Selection too small.")
            return
        
        t = sub_df['Time'].values
        i = sub_df['Current'].values
        v = sub_df['Voltage'].values
        
        # 2. Parse Params
        params = {}
        default_params = self.current_strategy.get_params()
        
        for key, widget in self.param_inputs.items():
            def_info = default_params[key]
            if def_info['type'] == 'bool':
                params[key] = widget.isChecked()
            else:
                try:
                    val_str = widget.text()
                    if def_info['type'] == 'float': params[key] = float(val_str)
                    else: params[key] = int(float(val_str))
                except ValueError:
                    widget.setText(str(def_info['default']))
                    QMessageBox.warning(self, "Invalid Input", f"Resetting {key}")
                    return

        # 3. Execute
        try:
            cycles, peak_indices = self.current_strategy.process(t, i, v, params)
            
            # --- VISUALIZE ---
            # A. Peaks on Detail Plot
            self.update_detail_view()
            if len(peak_indices) > 0:
                self.plot_detail.plot(t[peak_indices], i[peak_indices], pen=None, symbol='o', symbolBrush='r', symbolSize=8)
            
            # B. Strategy Specific Plot (Overlay vs Scatter)
            if not cycles:
                self.plot_result.clear()
                QMessageBox.information(self, "Result", "No cycles detected.")
                return
                
            self.current_strategy.visualize_results(self.plot_result, cycles, self.cmap_histogram)
            
        except Exception as e:
            QMessageBox.critical(self, "Analysis Failed", str(e))

if __name__ == "__main__":
    app = QApplication.instance()
    if not app: app = QApplication(sys.argv)
    w = BubbleCycleAnalyzer()
    w.show()
    app.exec()